<a href="https://colab.research.google.com/github/gkoushik03/AI_assignment/blob/main/reasoning_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reasoning agent: RuleTaker / CounterBench / FOLIO (fixed)

Run the cells **top to bottom**. Order matters: imports, data, engines A-B-C, router, evaluation, smoke tests, then the real run.

**What was fixed vs. the original notebook**
- Engine A called six helper functions that were never defined (`nl_parse`, `nl_closure`, `NL_CLOSED_WORLD`, `nl_clean`, `neg_split`, `holds_neg`) so every query raised `NameError`, which was swallowed, giving 0%. All are now implemented.
- `re`, `itertools`, `tqdm` were never imported before use. Now one imports cell.
- `EngineC` had no `.reason()` so it could not be routed; the router also referenced `engine_c` before it existed. All engines share one interface and are created before the router.
- Engine B: question parser handled only one direction of "X instead of Y"; silent overwrite of duplicate equations; unknown target defaulted to False; inconsistent observations answered "no"; enumeration is now limited to relevant ancestor variables.
- Engine C: digit-leading constants and `f(x) = y` now parse; inconsistent premises are flagged; one solver reused with push/pop.
- Evaluation: a single label normaliser, status / error / coverage reporting, accuracy by depth, and a loud warning when an engine answers (almost) nothing.
- Removed unused dicts, duplicated code and commented-out dead code; `datasets` is no longer shadowed.

In [1]:
# ==========================================
# SETUP: install + ALL imports in one place
# ==========================================
# (Previously `re`, `itertools` and `tqdm` were never imported before use.)
!pip install -q z3-solver

import re
import time
import itertools
from collections import Counter, defaultdict

import pandas as pd
import z3
from tqdm.auto import tqdm

In [3]:
# ==========================================
# LOAD DATASETS
# ==========================================
from datasets import load_dataset
from huggingface_hub import hf_hub_download, login


def try_hf_login(secret_names=("HF_TOKEN", "koushik")):
    """Optional login. These datasets are public, so failure is not fatal."""
    try:
        from google.colab import userdata          # only exists in Colab
    except ImportError:
        print("Not running in Colab - skipping HF login (not required for public data).")
        return
    for secret in secret_names:
        try:
            login(token=userdata.get(secret))
            print(f"Authenticated with Hugging Face Hub (secret: {secret}).")
            return
        except Exception:
            continue
    print("No HF token secret found - continuing anonymously.")


try_hf_login()


def load_counterbench(filename):
    path = hf_hub_download(repo_id="CounterBench/CounterBench",
                           filename=filename, repo_type="dataset")
    return load_dataset("json", data_files=path, split="train")


# NOTE: renamed from `datasets` so we no longer shadow the `datasets` library.
raw_splits = {
    "RuleTaker/train": load_dataset("jise/ruletaker", split="train"),
    "RuleTaker/test":  load_dataset("jise/ruletaker", split="test"),
    "RuleTaker/ood":   load_dataset("jise/ruletaker", split="ood_test"),
    "CounterBench/alpha":    load_counterbench("data_balanced_alpha_V1.json"),
    "CounterBench/backdoor": load_counterbench("data_balanced_backdoor_V2.json"),
}
folio = load_dataset("yale-nlp/FOLIO")
raw_splits["FOLIO/train"] = folio["train"]
raw_splits["FOLIO/validation"] = folio["validation"]

raw_splits = {name: ds.to_pandas() for name, ds in raw_splits.items()}

for name, df in raw_splits.items():
    print(f"{name:24s} shape={df.shape}  columns={df.columns.tolist()}")
print("\nAll seven dataset splits loaded successfully!")

Authenticated with Hugging Face Hub (secret: koushik).


folio_v2_train.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

folio_v2_validation.jsonl:   0%|          | 0.00/208k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1001 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/203 [00:00<?, ? examples/s]

RuleTaker/train          shape=(1000, 5)  columns=['context', 'statement', 'reasoning', 'depth', 'flag']
RuleTaker/test           shape=(250, 5)  columns=['context', 'statement', 'reasoning', 'depth', 'flag']
RuleTaker/ood            shape=(375, 5)  columns=['context', 'statement', 'reasoning', 'depth', 'flag']
CounterBench/alpha       shape=(1000, 6)  columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
CounterBench/backdoor    shape=(200, 6)  columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
FOLIO/train              shape=(1001, 7)  columns=['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']
FOLIO/validation         shape=(203, 7)  columns=['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']

All seven dataset splits loaded successfully!


In [4]:
# ==========================================
# STEP 1: DATASET INPUT PREPARATION
# ==========================================
ENGINE_BY_PREFIX = {
    "RuleTaker/":    "engine_a",
    "CounterBench/": "engine_b",
    "FOLIO/":        "engine_c",
}

# engine -> (context column, query column, ground-truth column)
COLUMN_MAP = {
    "engine_a": ("context",      "statement",      "flag"),
    "engine_b": ("given_info",   "question",       "answer"),
    "engine_c": ("premises-FOL", "conclusion-FOL", "label"),
}

reasoning_inputs = {}
engine_datasets = defaultdict(list)          # engine -> [split names]

for name, df in raw_splits.items():
    engine = next(e for prefix, e in ENGINE_BY_PREFIX.items() if name.startswith(prefix))
    ctx_col, qry_col, gt_col = COLUMN_MAP[engine]

    prepared = pd.DataFrame({
        "engine":            engine,
        "reasoning_context": df[ctx_col],
        "reasoning_query":   df[qry_col],
        "ground_truth":      df[gt_col],
    }, index=df.index)
    # keep RuleTaker depth so accuracy can be broken down per depth
    prepared["depth"] = df["depth"] if "depth" in df.columns else pd.NA

    reasoning_inputs[name] = prepared
    engine_datasets[engine].append(name)

print({e: names for e, names in engine_datasets.items()})

{'engine_a': ['RuleTaker/train', 'RuleTaker/test', 'RuleTaker/ood'], 'engine_b': ['CounterBench/alpha', 'CounterBench/backdoor'], 'engine_c': ['FOLIO/train', 'FOLIO/validation']}


In [5]:

import re
import itertools
from functools import lru_cache

# ==========================================
# ENGINE A: RULETAKER
# ==========================================

# ---------- Configuration ----------
NL_CLOSED_WORLD = True

# ---------- Words ----------
VAR_WORDS = {"someone", "somebody", "something", "anyone", "anybody", "anything",
             "everyone", "everybody", "everything"}
PRONOUNS = {"they", "it", "he", "she"}
GROUP_NOUNS = r"(?:people|persons|person|things|thing|animals|animal|objects|object|ones|one|someone|something|individuals)"
COPULAS = {"is", "are", "was", "were", "am"}
MODALS = {"can", "will", "could", "should", "would", "must", "may", "might", "shall"}

BLOCK_WORDS = re.compile(
    r"\b(or|either|neither|nor|causes?|caused|unless|only|some|no one|nobody|none)\b",
    re.I
)

COND = r"(?:if|when|whenever|assuming(?: that)?|based on(?: the fact that)?|given(?: that)?|since|provided(?: that)?|in case)"
CONC = (r"(?:then|we can conclude(?: that)?|we can say(?: that)?|it is evident that|it follows that|"
        r"this means(?: that)?|it means(?: that)?|this implies(?: that)?|which means(?: that)?|we know(?: that)?)")

RULE_RE1 = re.compile(rf"^{COND}\s+(.+?),\s+{CONC}\s+(.+)$", re.I)
RULE_RE2 = re.compile(r"^(?:if|when|whenever)\s+(.+?)\s+then\s+(.+)$", re.I)
REL_RE = re.compile(
    rf"^(?:all\s+|every\s+|any\s+)?{GROUP_NOUNS}\s+(?:who|that|which)\s+(.+?)\s+(are|is)\s+(.+)$",
    re.I
)
ALL_RE = re.compile(
    rf"^(all|every|each)\s+(.+?)\s+(?:{GROUP_NOUNS}\s+)?(are|is)\s+(.+)$",
    re.I
)
ADJ_RE = re.compile(
    rf"^(.+?)\s+(people|persons|things|animals|objects|ones|individuals)\s+(.+)$",
    re.I
)

HEADER_RE = re.compile(r"^\s*(?:facts|rules)\s*:", re.I)


# ---------- Text helpers ----------
def strip_label(s):
    return re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", s).strip()


def nl_clean(s):
    s = str(s).strip().rstrip(".?!").strip()
    return " ".join(s.split())


def singular_verb(w):
    w = w.lower()

    if w in ("are", "am"):
        return "is"
    if w == "were":
        return "was"
    if w == "have":
        return "has"
    if w == "do":
        return "does"
    if w in MODALS or w.endswith("s"):
        return w
    if re.search(r"(sh|ch|x|z|o)$", w):
        return w + "es"
    if re.search(r"[^aeiou]y$", w):
        return w[:-1] + "ies"

    return w + "s"


def split_subject(sentence):
    toks = sentence.split()

    if not toks:
        return "", ""

    low = [t.lower() for t in toks]

    if low[0] in VAR_WORDS or low[0] in PRONOUNS or low[0].startswith("?"):
        k = 1

    elif low[0] in ("the", "a", "an"):
        k = 2

        while k < len(toks) and low[k] not in COPULAS and low[k] not in MODALS and not \
                (low[k].endswith("s") and k >= 2) and low[k] not in ("does", "do", "has", "have"):
            k += 1

    else:
        k = 1

        while k < len(toks) and toks[k][:1].isupper():
            k += 1

    return " ".join(low[:k]), " ".join(low[k:])


def neg_split(atom):
    a = " " + atom + " "

    for neg, pos in [
        (" is not ", " is "), (" isn't ", " is "),
        (" are not ", " are "), (" aren't ", " are "),
        (" was not ", " was "), (" were not ", " were ")
    ]:
        if neg in a:
            return True, a.replace(neg, pos, 1).strip()

    m = re.search(r" (?:does not|doesn't|do not|don't|did not|didn't) (\w+)", a)

    if m:
        return True, (
            a[:m.start()] + " " + singular_verb(m.group(1)) + a[m.end():]
        ).strip()

    return False, atom


def to_template(text, has_var):
    toks = nl_clean(text).split()
    out = []

    for i, t in enumerate(toks):
        lt = t.lower()

        if lt in VAR_WORDS:
            out.append("?x")

        elif has_var and lt in PRONOUNS:
            out.append("?x")

            if lt == "they" and i + 1 < len(toks):
                toks[i + 1] = singular_verb(toks[i + 1])

        elif has_var and lt in ("them", "him", "her") and i > 0:
            out.append("?x")

        else:
            out.append(lt)

    return " ".join(out)


def complete_part(part, first_words):
    w = part.split()[0] if part.split() else ""

    return (
        w[:1].isupper()
        or w.lower() in ("the", "a", "an")
        or w.lower() in VAR_WORDS
        or w.lower() in PRONOUNS
        or w.lower() in first_words
    )


def split_conditions(cond_text, first_words, vocab):
    parts = [
        p for p in re.split(r",\s*(?:and\s+)?|\s+and\s+", cond_text)
        if p.strip()
    ]

    if not parts:
        return []

    subj, rest = split_subject(parts[0])
    verb = rest.split()[0] if rest.split() else ""

    out = [parts[0]]

    for p in parts[1:]:
        if complete_part(p, first_words):
            out.append(p)
            continue

        cands = [f"{subj} {p}"]

        if verb in COPULAS:
            cands.insert(0, f"{subj} {verb} {p}")

        pick = next(
            (c for c in cands if split_subject(c)[1].lower() in vocab),
            cands[0]
        )

        out.append(pick)

    return out


# ---------- Rule representation ----------
class NLRule:
    def __init__(self, conds, concl, text):
        self.conds = conds
        self.concl = concl
        self.text = text


# ---------- Universal rules ----------
IRREGULAR = {
    "men": "man", "women": "woman", "people": "person",
    "children": "child", "mice": "mouse", "geese": "goose",
    "feet": "foot", "teeth": "tooth"
}


def noun_form(w):
    w = w.lower()

    if w in IRREGULAR:
        sing = IRREGULAR[w]
    elif w.endswith("ies") and len(w) > 4:
        sing = w[:-3] + "y"
    elif w.endswith("s") and not re.search(r"(ss|us|ous|is)$", w):
        sing = w[:-1]
    else:
        return None

    return ("an " if sing[0] in "aeiou" else "a ") + sing


def class_atom(words, is_rel=False):
    toks = words.lower().split()

    if not toks:
        return None

    if toks[0] in COPULAS:
        return "?x is " + " ".join(toks[1:])

    if is_rel:
        return "?x " + singular_verb(toks[0]) + (
            " " + " ".join(toks[1:]) if len(toks) > 1 else ""
        )

    return "?x is " + " ".join(toks)


def parse_universal(s):
    m = REL_RE.match(s)

    if m:
        conds = [
            class_atom(c, True)
            for c in re.split(r"\s+and\s+", m.group(1))
        ]
        return conds, "?x is " + m.group(3).lower()

    m = ALL_RE.match(s)

    if m:
        adjs = [
            a for a in re.split(r",\s*|\s+and\s+", m.group(2))
            if a.strip()
        ]

        head = m.group(4).lower()

        no_group = not re.search(
            GROUP_NOUNS + r"\s+(are|is)\s", s, re.I
        )

        if no_group and len(adjs) == 1 and len(adjs[0].split()) == 1 and noun_form(adjs[0]):
            nf = noun_form(head) if len(head.split()) == 1 else None

            return [
                "?x is " + noun_form(adjs[0])
            ], "?x is " + (nf or head)

        return [
            "?x is " + a.lower() for a in adjs
        ], "?x is " + head

    m = ADJ_RE.match(s)

    if m:
        adj_part, rest = m.group(1), m.group(3).split()

        words = re.split(
            r",\s*|\s+and\s+|\s+", adj_part.lower()
        )

        if (
            rest
            and rest[0].lower() in ("are", "is")
            and len(words) <= 4
            and not set(words) & (
                {"the", "a", "an", "who", "that", "which", "of"} | COPULAS
            )
            and not any(w[:1].isupper() for w in adj_part.split()[1:])
        ):
            adjs = [
                a for a in re.split(r",\s*|\s+and\s+", adj_part)
                if a.strip()
            ]

            return [
                "?x is " + a.lower() for a in adjs
            ], "?x is " + " ".join(rest[1:]).lower()

    return None


# ---------- Sentence parsing ----------
def nl_sentences(context):
    text = "\n".join(
        l for l in context.split("\n")
        if not HEADER_RE.match(l)
    )

    text = re.sub(
        r"^\s*(facts|rules)\s*:\s*", "", text,
        flags=re.I | re.M
    )

    out = []

    for line in text.split("\n"):
        line = strip_label(line)

        for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", line):
            s = nl_clean(s)

            if s:
                out.append(s)

    return out


def nl_parse(context, statement=""):
    sents = nl_sentences(context)

    if not sents:
        return None

    raw_rules, facts_txt = [], []

    has_headers = bool(
        re.search(r"^\s*(facts|rules)\s*:", context, re.I | re.M)
    )

    for s in sents:
        m = RULE_RE1.match(s) or RULE_RE2.match(s)

        if m:
            raw_rules.append((m.group(1), m.group(2), s))
            continue

        u = parse_universal(s)

        if u:
            raw_rules.append(("__universal__", u, s))
            continue

        if "," in s or " causes " in s or " cause " in s:
            return None

        if not has_headers and BLOCK_WORDS.search(s):
            return None

        facts_txt.append(s)

    first_words = {
        f.split()[0].lower()
        for f in facts_txt if f.split()
    }

    vocab = (
        {split_subject(f)[1] for f in facts_txt}
        | {split_subject(nl_clean(statement))[1]}
    )

    for c, d, _ in raw_rules:
        if c != "__universal__":
            vocab.add(split_subject(nl_clean(d))[1])

    vocab |= {
        singular_verb(v.split()[0]) + v[len(v.split()[0]):]
        for v in list(vocab) if v.split()
    }

    facts, rules, entities = set(), [], set()

    for f in facts_txt:
        low = f.lower()

        if set(low.split()) & VAR_WORDS:
            neg, atom = neg_split(to_template(f, True))
            rules.append(NLRule([], (neg, atom), f))
            continue

        neg, atom = neg_split(nl_clean(low))
        facts.add(("NOT:" if neg else "") + atom)
        entities.add(split_subject(f)[0])

        if " and " in low:
            for part in split_conditions(f, first_words, vocab):
                n2, a2 = neg_split(nl_clean(part.lower()))
                facts.add(("NOT:" if n2 else "") + a2)

    for cond, concl, text in raw_rules:
        if cond == "__universal__":
            conds, head = concl

            rules.append(
                NLRule(
                    [neg_split(c) for c in conds if c],
                    neg_split(head),
                    text
                )
            )
            continue

        parts = split_conditions(cond, first_words, vocab)

        has_var = bool(
            set(" ".join(parts + [concl]).lower().split()) & VAR_WORDS
        )

        conds = [
            neg_split(to_template(p, has_var))
            for p in parts
        ]

        rules.append(
            NLRule(
                conds,
                neg_split(to_template(concl, has_var)),
                text
            )
        )

    if not rules and not facts:
        return None

    q_subj = split_subject(nl_clean(statement))[0]

    if q_subj:
        entities.add(q_subj)

    return facts, rules, {e for e in entities if e}


# ---------- Variable matching ----------
@lru_cache(maxsize=4096)
def tmpl_regex(t):
    rx, seen = "", set()

    for p in re.split(r"(\?[a-z]\w*)", t):
        if p.startswith("?"):
            n = p[1:]

            rx += (
                f"(?P={n})"
                if n in seen
                else f"(?P<{n}>.+?)"
            )

            seen.add(n)
        else:
            rx += re.escape(p)

    return re.compile(rx)


def subst(t, b):
    return re.sub(
        r"\?([a-z]\w*)",
        lambda m: b.get(m.group(1), m.group(0)),
        t
    ) if "?" in t else t


def has_vars(t):
    return "?" in t


def holds_neg(atom, facts, cwa):
    return (
        ("NOT:" + atom) in facts
        or (cwa and atom not in facts)
    )


def rule_bindings(rule, facts, entities, cwa):
    """All variable bindings that make every condition of the rule true."""

    pos = [t for n, t in rule.conds if not n]
    neg = [t for n, t in rule.conds if n]

    def match_pos(i, b):
        if i == len(pos):
            yield b
            return

        t = subst(pos[i], b)

        if not has_vars(t):
            if t in facts:
                yield from match_pos(i + 1, b)
            return

        rx = tmpl_regex(t)

        for f in facts:
            if f.startswith("NOT:"):
                continue

            m = rx.fullmatch(f)

            if m:
                yield from match_pos(
                    i + 1,
                    {**b, **m.groupdict()}
                )

    for b in match_pos(0, {}):
        free = sorted({
            v
            for t in neg + [rule.concl[1]]
            for v in re.findall(
                r"\?([a-z]\w*)", subst(t, b)
            )
        })

        for combo in itertools.product(
            sorted(entities), repeat=len(free)
        ):
            bb = {**b, **dict(zip(free, combo))}

            if all(
                holds_neg(subst(t, bb), facts, cwa)
                for t in neg
            ):
                yield bb


# ---------- Forward chaining ----------
def nl_closure(facts, rules, entities, cwa):
    """Forward chaining. Negated conditions run after positive rules settle."""

    facts, trace = set(facts), []

    pos_rules = [
        r for r in rules
        if not any(n for n, _ in r.conds)
    ]

    naf_rules = [
        r for r in rules
        if any(n for n, _ in r.conds)
    ]

    def fire(rule_list, naf):
        added = False

        for r in rule_list:
            for b in list(
                rule_bindings(
                    r, facts, entities,
                    cwa if naf else False
                )
            ):
                neg, t = r.concl

                new = (
                    ("NOT:" if neg else "")
                    + subst(t, b)
                )

                if "?" not in new and new not in facts:
                    facts.add(new)
                    trace.append(
                        f"{r.text}   =>   {new}"
                    )
                    added = True

        return added

    for _ in range(1000):
        while fire(pos_rules, False):
            pass

        if not naf_rules or not fire(naf_rules, True):
            break

    return facts, trace


# ---------- Original RuleTaker interface ----------
def nl_try(context, statement):
    """Returns (answer, trace) or None if parsing fails."""

    parsed = nl_parse(context, statement)

    if parsed is None or not statement.strip():
        return None

    facts, rules, entities = parsed

    all_facts, trace = nl_closure(
        facts, rules, entities, NL_CLOSED_WORLD
    )

    neg, q = neg_split(nl_clean(statement).lower())

    if neg:
        ans = holds_neg(q, all_facts, NL_CLOSED_WORLD)
    else:
        ans = q in all_facts

    trace.append(
        f"query '{'not ' if neg else ''}{q}' -> {ans}"
    )

    return ans, trace


# ==========================================
# ENGINE A CLASS
# ==========================================
class RuleTakerEngine:

    def __init__(self):
        self.name = "Engine A - RuleTaker"

    def reason(self, context, statement):
        try:
            result = nl_try(context, statement)

            if result is None:
                return {
                    "engine": self.name,
                    "status": "parse_failed",
                    "prediction": None,
                    "trace": [],
                    "error": "Could not parse context or statement"
                }

            prediction, trace = result

            return {
                "engine": self.name,
                "status": "success",
                "prediction": prediction,
                "trace": trace,
                "error": None
            }

        except Exception as e:
            return {
                "engine": self.name,
                "status": "error",
                "prediction": None,
                "trace": [],
                "error": f"{type(e).__name__}: {e}"
            }


# ---------- Create Engine A ----------
engine_a = RuleTakerEngine()


In [6]:
# ==========================================
# ENGINE B: COUNTERBENCH CAUSAL REASONING
# ==========================================
import re
import itertools

CB_MAX_ROOTS = 18
CB_THRESHOLD = 0.5


# ---------- literal handling ----------
def read_literal(text):
    text = text.strip()

    # Negative forms
    if re.match(r"^not\s+", text, re.I):
        text = re.sub(r"^not\s+", "", text, flags=re.I)
        text = re.sub(r"\s+occurs?$", "", text, flags=re.I)
        return text.strip(), False

    if re.search(r"\s+(does not|do not|is not|are not|not occurs?)$", text, re.I):
        text = re.sub(
            r"\s+(does not|do not|is not|are not|not occurs?)$",
            "", text, flags=re.I
        )
        return text.strip(), False

    text = re.sub(r"\s+occurs?$", "", text, flags=re.I)
    return text.strip(), True

def split_list(text):
    return [p.strip() for p in re.split(r",\s*|\s+and\s+", text, flags=re.I) if p.strip()]


# ---------- parse causal information ----------
def cb_parse_info(info):
    sentences = [s.strip() for s in re.split(r"\.(?!\d)\s*", info) if s.strip()]
    equations, observed, priors = {}, {}, {}

    for s in sentences:
        m = re.match(r"^We observed\s+(.+)$", s, re.I)
        if m:
            for item in split_list(m.group(1)):
                var, value = read_literal(item)
                if var in observed and observed[var] != value:
                    raise ValueError(f"Contradictory observations for {var}")
                observed[var] = value
            continue

        m = re.match(r"^(\w+)\s*~\s*Bern\(([\d.]+)\)$", s, re.I)
        if m:
            var, p = m.group(1), float(m.group(2))
            if not 0 <= p <= 1:
                raise ValueError(f"Invalid prior probability for {var}: {p}")
            priors[var] = p
            continue

        s = re.sub(r"^We know that\s+", "", s, flags=re.I)
        clauses = []

        for frag in s.split(","):
            frag = re.sub(r"^and\s+", "", frag.strip(), flags=re.I)
            if re.search(r"\bcauses?\b", frag, re.I) or not clauses:
                clauses.append(frag)
            else:
                clauses[-1] += ", " + frag

        for c in clauses:
            m = re.match(r"^(.*?)\s+(?:together\s+)?causes?\s+(.*)$", c, re.I)
            if not m:
                raise ValueError("Cannot parse clause: " + c)

            cause, effect = m.group(1).strip(), m.group(2).strip()

            if re.search(r"\s+or\s+", cause, re.I):
                op = "or"
                inputs = [read_literal(x) for x in re.split(r"\s+or\s+", cause, flags=re.I)]
            else:
                op = "and"
                cause = re.sub(r"\s+together$", "", cause, flags=re.I)
                inputs = [read_literal(x) for x in re.split(r"\s+and\s+", cause, flags=re.I)]

            for item in split_list(effect):
                var, sign = read_literal(item)
                new_eq = (op, inputs, sign)

                if var in equations and equations[var] != new_eq:
                    raise ValueError(f"Multiple or conflicting equations for {var}")

                equations[var] = new_eq

    if not equations:
        raise ValueError("No cause sentences found")

    return equations, observed, priors


# ---------- parse counterfactual question ----------
def cb_parse_question(q):
    q = q.strip()

    # Would C occur if A instead of not A?
    m = re.match(r"^Would\s+(\w+)\s+occur\s+if\s+(.+?)\s+instead\s+of\s+(.+?)\?$", q, re.I)
    if m:
        target = m.group(1)
        var1, value1 = read_literal(m.group(2))
        var2, value2 = read_literal(m.group(3))

        if var1 != var2 or value1 == value2:
            raise ValueError("The two 'instead of' alternatives must be opposite values of the same variable")

        return target, {var1: value1}

    # Assume A, and based on this assumption, further suppose B. Would C occur?
    m = re.match(
        r"^Assume\s+(.+?),\s*and based on this assumption,\s*"
        r"further suppose\s+(.+?)\.\s*Would\s+(\w+)\s+occur\?$",
        q, re.I
    )
    if m:
        a = read_literal(m.group(1))
        b = read_literal(m.group(2))

        if a[0] == b[0] and a[1] != b[1]:
            raise ValueError(f"Contradictory assumptions for {a[0]}")

        return m.group(3), {a[0]: a[1], b[0]: b[1]}

    # Would C occur if A and B?
    m = re.match(r"^Would\s+(\w+)\s+occur\s+if\s+(.+)\?$", q, re.I)
    if m:
        changes = {}
        for item in split_list(m.group(2)):
            var, value = read_literal(item)
            if var in changes and changes[var] != value:
                raise ValueError(f"Contradictory interventions for {var}")
            changes[var] = value
        return m.group(1), changes

    raise ValueError("Cannot parse question: " + q)


# ---------- causal graph ----------
def cb_ancestors(equations, seeds):
    seen, stack = set(), list(seeds)

    while stack:
        v = stack.pop()
        if v in seen:
            continue

        seen.add(v)

        if v in equations:
            stack.extend(x for x, _ in equations[v][1])

    return seen


def cb_topo_order(equations, variables):
    order, state = [], {}

    def visit(v):
        if state.get(v) == 2:
            return
        if state.get(v) == 1:
            raise ValueError("Cyclic causal graph")

        state[v] = 1

        if v in equations:
            for x, _ in equations[v][1]:
                visit(x)
            order.append(v)

        state[v] = 2

    for v in sorted(variables):
        visit(v)

    return order


# ---------- evaluate causal model ----------
def compute_all(equations, order, root_values, forced):
    values = dict(root_values)
    values.update(forced)

    for var in order:
        if var in forced:
            continue

        op, inputs, sign = equations[var]
        bits = [values[x] == s for x, s in inputs]
        result = any(bits) if op == "or" else all(bits)
        values[var] = result if sign else not result

    return values


# ---------- abduction -> action -> prediction ----------
def cb_solve(context, statement):
    if (not re.search(r"\bcauses?\b", context, re.I)
            or not re.match(r"^\s*(would|assume)\b", statement, re.I)):
        return {
            "status": "parse_failed",
            "prediction": None,
            "trace": [],
            "error": "Not a causal-story / counterfactual-question pair"
        }

    try:
        equations, observed, priors = cb_parse_info(context)
        target, changes = cb_parse_question(statement)

        relevant = cb_ancestors(
            equations,
            set(observed) | {target} | set(changes)
        )

        roots = sorted(v for v in relevant if v not in equations)
        order = cb_topo_order(equations, relevant)

        for var in priors:
            if var in equations:
                raise ValueError(f"Prior specified for non-root variable: {var}")

    except Exception as e:
        return {
            "status": "parse_failed",
            "prediction": None,
            "trace": [],
            "error": f"{type(e).__name__}: {e}"
        }

    if len(roots) > CB_MAX_ROOTS:
        return {
            "status": "too_many_roots",
            "prediction": None,
            "trace": [],
            "error": f"{len(roots)} relevant roots > CB_MAX_ROOTS={CB_MAX_ROOTS}"
        }

    yes_w = total_w = 0.0
    worlds = 0

    for combo in itertools.product([True, False], repeat=len(roots)):
        root_values = dict(zip(roots, combo))

        # Abduction: retain worlds matching observations
        real = compute_all(equations, order, root_values, {})

        if any(real.get(v) != val for v, val in observed.items()):
            continue

        # Probability of this exogenous world
        w = 1.0
        for r, val in root_values.items():
            p = priors.get(r, 0.5)
            w *= p if val else (1 - p)

        # Action: apply intervention
        counterfactual = compute_all(equations, order, root_values, changes)

        worlds += 1
        total_w += w

        if counterfactual.get(target, False):
            yes_w += w

    if total_w == 0:
        return {
            "status": "inconsistent_observations",
            "prediction": None,
            "trace": [],
            "error": "No world with positive probability is consistent with the observations"
        }

    p_yes = yes_w / total_w
    answer = p_yes > CB_THRESHOLD

    trace = [
        f"Root variables: {roots}",
        f"Observed facts: {observed}",
        f"Intervention: {changes}",
        f"Worlds matching observations: {worlds}",
        f"P({target}=True | intervention, observations) = {p_yes:.4f}",
        f"Threshold = {CB_THRESHOLD} -> prediction = {answer}"
    ]

    return {
        "status": "success",
        "prediction": answer,
        "probability": p_yes,
        "target": target,
        "intervention": changes,
        "worlds": worlds,
        "trace": trace,
        "error": None
    }


class CounterBenchEngine:
    def __init__(self):
        self.name = "Engine B - CounterBench"

    def reason(self, context, statement):
        try:
            return cb_solve(context, statement)
        except Exception as e:
            return {
                "status": "error",
                "prediction": None,
                "trace": [],
                "error": f"{type(e).__name__}: {e}"
            }


engine_b = CounterBenchEngine()

In [7]:
DEFAULT_ANSWER  = False
DPLL_MAX_STEPS  = 20000
GROUND_LIMIT    = 300000
import re, sys, itertools
from functools import lru_cache

sys.setrecursionlimit(10000)   # DPLL is recursive

FOL_SYMBOLS = set("∀∃¬∧∨→↔⊕")

HEADER_RE = re.compile(r"^\s*(facts?|rules?|premises?|axioms?|knowledge\s*base|kb|theory|context|logic|"
                       r"given|background)\s*:?\s*$", re.I)
# "1. ", "2) ", "- ", "Fact 1: ", "Rule: ", "P1: "
LABEL_RE = re.compile(r"^\s*(?:(?:fact|rule|premise|axiom)s?\s*\d*\s*[:.)\-]|[prf]\d+\s*[:.)]|\d+\s*[.):]|[-*•](?=\s))\s*", re.I)   # a bullet needs a space after it ("-P(x)" is "not P(x)")
# "Query: ...", "Question: ...", "?- ..."
QUERY_LINE_RE = re.compile(r"^\s*(?:(?:query|question|statement|goal|hypothesis|conclusion|prove|test)\s*\d*\s*:|\?-)\s*(.*)$", re.I)
ATOM_RE = re.compile(r"[A-Za-z_][\w'’.\-]*\s*\(")


def to_text(x):
    """Any cell value (NaN, list, dict, string that looks like a list) -> plain text."""
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    if isinstance(x, (list, tuple)):
        return "\n".join(to_text(i) for i in x)
    if isinstance(x, dict):
        return "\n".join(to_text(v) for v in x.values())
    s = str(x)
    t = s.strip()
    if t.startswith("[") and t.endswith("]") and ("'" in t or '"' in t):
        try:
            v = ast.literal_eval(t)
            if isinstance(v, (list, tuple)):
                return to_text(v)
        except Exception:
            pass
    return s


def prepare(context, statement):
    """Clean one question. If the query is written inside the context, move it out."""
    c, s = to_text(context).replace("\r", ""), to_text(statement).replace("\r", "").strip()
    keep = []
    for line in c.split("\n"):
        m = QUERY_LINE_RE.match(line)
        if m and not s:
            s = m.group(1).strip()
            continue
        keep.append(line)
    c = "\n".join(keep)
    m = QUERY_LINE_RE.match(s)
    if m:
        s = m.group(1).strip()
    return c, s


def strip_label(line):
    return LABEL_RE.sub("", line, count=1).strip()


# ---------------- logic text -> standard symbols ----------------

def looks_logical(text):
    """True if the text is written in logic (any common style), not in English."""
    if set(text) & FOL_SYMBOLS or ":-" in text:
        return True
    lines = [strip_label(l) for l in text.split("\n") if l.strip() and not HEADER_RE.match(l)]
    if not lines:
        return False
    has_ops = re.search(r"->|=>|<->|<=>|&|\||~|\\\+|\b(forall|exists)\b", text, re.I)
    if has_ops and ATOM_RE.search(text):
        return True
    # every line is just atoms, e.g. "Young(bob)." or "smart(X)"
    atom_line = re.compile(r"^[¬~!\-\s]*[A-Za-z_][\w'’.\-]*\s*\([^()]*\)\s*[.;]?$")
    return all(atom_line.match(l) for l in lines)


def ascii_to_fol(t):
    """Convert ASCII / Prover9 / textbook styles into the symbols the parser uses."""
    t = t.replace("[", "(").replace("]", ")").replace("{", "(").replace("}", ")")
    t = t.replace('"', "").replace("`", "")
    for a, b in [("<=>", "↔"), ("<->", "↔"), ("⟷", "↔"), ("⇔", "↔"), ("≡", "↔"),
                 ("-->", "→"), ("->", "→"), ("=>", "→"), ("⇒", "→"), ("⊃", "→"),
                 ("!=", "≠"), ("\\=", "≠"), ("&&", "∧"), ("&", "∧"), ("^", "∧"), ("·", "∧"),
                 ("||", "∨"), ("|", "∨"), ("\\+", "¬"), ("~", "¬"), ("∼", "¬"), ("!", "¬"),
                 ("⊻", "⊕"), ("Ɐ", "∀")]:
        t = t.replace(a, b)
    word_ops = [(r"\bforall\b", "∀"), (r"\bexists\b", "∃"), (r"\bnot\b", "¬"), (r"\band\b", "∧"),
                (r"\bor\b", "∨"), (r"\bxor\b", "⊕"), (r"\bimplies\b", "→"), (r"\biff\b", "↔")]
    for a, b in word_ops:
        t = re.sub(a, b, t, flags=re.I)
    # Prover9 style "all x" / "exists x"
    t = re.sub(r"\b(?:all|every)\s+([A-Za-z]\w*)\s*(?=[.:,(∀∃¬]|\s)", r"∀\1 ", t, flags=re.I)
    # "-P(x)" means not P(x)   (a hyphen inside names like self-driving is kept)
    t = re.sub(r"(?<![\w)\-])-(?=\s*[A-Za-z(¬∀∃])", "¬", t)
    # "∀x,y (" -> "∀x ∀y ("
    t = re.sub(r"([∀∃])\s*([A-Za-z]\w*(?:\s*,\s*[A-Za-z]\w*)+)(?=\s*[(.:])",
               lambda m: " ".join(m.group(1) + v.strip() for v in m.group(2).split(",")), t)
    # "∀x." / "∀x:" / "∀ x," -> "∀x "
    t = re.sub(r"([∀∃])\s*([A-Za-z]\w*)\s*[.:,]\s*", r"\1\2 ", t)
    return t


def split_top(s, seps):
    """Split on separators that are not inside brackets."""
    parts, depth, cur = [], 0, ""
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        if ch in seps and depth == 0:
            parts.append(cur)
            cur = ""
        else:
            cur += ch
    parts.append(cur)
    return [p.strip() for p in parts if p.strip()]


def prolog_clause_to_fol(clause):
    """'successful(X) :- smart(X), kind(X)'  ->  '∀x1 (smart(x1) ∧ kind(x1) → successful(x1))'"""
    clause = clause.strip().rstrip(".").strip()
    vars_ = []
    for v in re.findall(r"(?<![\w'])([A-Z_]\w*)(?=\s*[,)])", clause):
        if v not in vars_:
            vars_.append(v)
    for i, v in enumerate(vars_):
        clause = re.sub(rf"(?<![\w']){re.escape(v)}(?=\s*[,)])", f"pv{i+1}", clause)

    def lit(x):
        x = x.strip()
        if x.startswith("\\+"):
            return "¬" + x[2:].strip()
        m = re.match(r"^not\s*\((.*)\)$", x)
        if m:
            return "¬" + m.group(1)
        if x.startswith("not "):
            return "¬" + x[4:]
        return x

    def conj(x):
        alts = split_top(x, ";")
        out = ["(" + " ∧ ".join(lit(a) for a in split_top(alt, ",")) + ")" for alt in alts]
        return out[0] if len(out) == 1 else "(" + " ∨ ".join(out) + ")"

    if ":-" in clause:
        head, body = clause.split(":-", 1)
        f = f"({conj(body)} → {conj(head)})"
    else:
        f = conj(clause)
    for i in reversed(range(len(vars_))):
        f = f"∀pv{i+1} {f}"
    return f


def logic_lines(context):
    """Split a logic context into one formula per item."""
    is_prolog = ":-" in context
    out = []
    for line in context.split("\n"):
        if not line.strip() or HEADER_RE.match(line):
            continue
        line = strip_label(line)
        if not line:
            continue
        if is_prolog:
            for cl in split_top(line, "."):
                out.append(prolog_clause_to_fol(cl))
        else:
            # several formulas on one line: "A(b). B(c)."  or  "A(b); B(c)"
            for part in re.split(r"(?<=\))\s*[.;]\s+(?=\S)", line):
                if part.strip():
                    out.append(ascii_to_fol(part.strip()))
    return out


def logic_query(statement, is_prolog=False):
    s = statement.strip().rstrip("?").strip().rstrip(".").strip()
    if is_prolog:
        f = prolog_clause_to_fol(s)
        return f.replace("∀", "∃")           # Prolog query variables mean "some"
    return ascii_to_fol(s)

TOKEN_RE = re.compile(r"\s*(∀|∃|¬|∧|∨|→|↔|⟷|⊕|≠|=|:|\(|\)|,|[\w’'.\-]+)")

def tokenize(text):
    text = text.strip().replace("—>", "→").rstrip(".")
    tokens, pos = [], 0
    while pos < len(text):
        m = TOKEN_RE.match(text, pos)
        if not m:
            raise ValueError(f"Bad character at {pos}: {text[pos:]}")
        tok = m.group(1)
        tokens.append("↔" if tok == "⟷" else tok)
        pos = m.end()
        while pos < len(text) and text[pos].isspace():
            pos += 1
    fixed, depth = [], 0                      # repair unbalanced brackets
    for tok in tokens:
        if tok == ")":
            if depth == 0:
                continue
            depth -= 1
        elif tok == "(":
            depth += 1
        fixed.append(tok)
    return fixed + [")"] * depth

class Parser:
    """Precedence (strong -> weak):  ¬  ∧  ∨  ⊕  →  ↔"""
    def __init__(self, tokens):
        self.t, self.i = tokens, 0
    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else None
    def take(self, expected=None):
        tok = self.peek()
        if tok is None and expected is None:
            raise ValueError("Unexpected end of formula")
        if expected and tok != expected:
            raise ValueError(f"Expected {expected}, got {tok}")
        self.i += 1
        return tok
    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise ValueError("Extra tokens: " + " ".join(self.t[self.i:]))
        return f
    def iff(self):
        left = self.imp()
        while self.peek() == "↔":
            self.take(); left = ("iff", left, self.imp())
        return left
    def imp(self):
        left = self.xor()
        if self.peek() == "→":
            self.take(); return ("imp", left, self.imp())
        return left
    def xor(self):
        left = self.or_()
        while self.peek() == "⊕":
            self.take(); left = ("xor", left, self.or_())
        return left
    def or_(self):
        left = self.and_()
        while self.peek() == "∨":
            self.take(); left = ("or", left, self.and_())
        return left
    def and_(self):
        left = self.unary()
        while self.peek() == "∧":
            self.take(); left = ("and", left, self.unary())
        return left
    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.take(); return ("not", self.unary())
        if tok in ("∀", "∃"):
            self.take()
            var = norm_name(self.take() or "")
            if self.peek() in (",", ":", "."):
                self.take()
            body = self.paren() if self.peek() == "(" else self.unary()
            return ("all" if tok == "∀" else "ex", var, body)
        if tok == "(":
            return self.paren()
        return self.atom()
    def paren(self):
        self.take("(")
        f = self.iff()
        if self.peek() == ")":
            self.take()
        return f
    def atom(self):
        name = self.take()
        if name is None:
            raise ValueError("Unexpected end of formula")
        if self.peek() in ("=", "≠"):
            op = self.take()
            eq = ("atom", "=", (norm_name(name), norm_name(self.take() or "")))
            return eq if op == "=" else ("not", eq)
        args = []
        if self.peek() == "(":
            self.take()
            while self.peek() not in (")", None):
                args.append(norm_name(self.take()))
                if self.peek() == ",":
                    self.take()
            self.take(")")
        if name in ("∀", "∃", "¬", "∧", "∨", "→", "↔", "⊕", "(", ")", ",") :
            raise ValueError("Unexpected symbol " + name)
        return ("atom", norm_name(name), tuple(args))

def norm_name(s):
    """Bob / bob / 'bob' / is_young / IsYoung -> same name"""
    return s.lower().replace("_", "").strip("'’\"")

@lru_cache(maxsize=20000)
def parse_fol(text):
    return Parser(tokenize(text)).parse()

VAR_NAMES = {"x", "y", "z", "w", "v", "u", "x1", "x2", "y1", "y2"}

def bound_vars_used(f, bound):
    k = f[0]
    if k == "atom":
        return {a for a in f[2] if a in bound}
    if k == "not":
        return bound_vars_used(f[1], bound)
    if k in ("all", "ex"):
        return bound_vars_used(f[2], bound - {f[1]})
    return bound_vars_used(f[1], bound) | bound_vars_used(f[2], bound)

def close(f):
    for v in sorted(bound_vars_used(f, VAR_NAMES)):       # free variable -> "for all"
        f = ("all", v, f)
    return f

class Names:
    def __init__(self):
        self.n = 0
    def new(self, prefix):
        self.n += 1
        return f"{prefix}{self.n}"

def name_it(f, names, defs, bound):
    small = f[0] == "atom" or (f[0] == "not" and f[1][0] == "atom")
    if small:
        return f
    vs = tuple(sorted(bound_vars_used(f, set(bound))))
    new_atom = ("atom", names.new("aux"), vs)
    d = ("iff", new_atom, f)
    for v in vs:
        d = ("all", v, d)
    defs.append(d)
    return new_atom

def shorten(f, names, defs, bound=frozenset()):
    """Inside ⊕ / ↔, replace big parts by new atoms (avoids exponential growth)."""
    k = f[0]
    if k == "atom":
        return f
    if k == "not":
        return ("not", shorten(f[1], names, defs, bound))
    if k in ("all", "ex"):
        return (k, f[1], shorten(f[2], names, defs, bound | {f[1]}))
    a = shorten(f[1], names, defs, bound)
    b = shorten(f[2], names, defs, bound)
    if k in ("xor", "iff"):
        a = name_it(a, names, defs, bound)
        b = name_it(b, names, defs, bound)
    return (k, a, b)

def nnf(f, pos=True):
    k = f[0]
    if k == "atom":
        return ("lit", pos, f[1], f[2])
    if k == "not":
        return nnf(f[1], not pos)
    if k in ("and", "or"):
        a, b = nnf(f[1], pos), nnf(f[2], pos)
        return ("and", a, b) if (k == "and") == pos else ("or", a, b)
    if k == "imp":
        return nnf(("or", ("not", f[1]), f[2]), pos)
    if k == "iff":
        a, b = f[1], f[2]
        return nnf(("and", ("or", ("not", a), b), ("or", a, ("not", b))), pos)
    if k == "xor":
        a, b = f[1], f[2]
        return nnf(("and", ("or", a, b), ("or", ("not", a), ("not", b))), pos)
    if k == "all":
        return ("all" if pos else "ex", f[1], nnf(f[2], pos))
    if k == "ex":
        return ("ex" if pos else "all", f[1], nnf(f[2], pos))
    raise ValueError(k)

def skolemize(f, names, univ=(), sub=None):
    sub = sub or {}
    k = f[0]
    if k == "lit":
        return ("lit", f[1], f[2], tuple(sub.get(a, a) for a in f[3]))
    if k in ("and", "or"):
        return (k, skolemize(f[1], names, univ, sub), skolemize(f[2], names, univ, sub))
    if k == "all":
        v = "?" + names.new("v")
        return skolemize(f[2], names, univ + (v,), {**sub, f[1]: v})
    if k == "ex":
        sk = names.new("sk")
        term = sk if not univ else sk + "(" + ",".join(univ) + ")"
        return skolemize(f[2], names, univ, {**sub, f[1]: term})
    raise ValueError(k)

def cnf(f):
    k = f[0]
    if k == "lit":
        return [[(f[1], f[2], f[3])]]
    if k == "and":
        return cnf(f[1]) + cnf(f[2])
    if k == "or":
        return [a + b for a in cnf(f[1]) for b in cnf(f[2])]
    raise ValueError(k)

def to_clauses(formula, names):
    defs = []
    main = shorten(close(formula), names, defs)
    clauses = []
    for f in [main] + defs:
        clauses += cnf(skolemize(nnf(f), names))
    return clauses

VAR_RE = re.compile(r"\?v\d+")

def constants_in(clauses):
    consts = set()
    for c in clauses:
        for _, _, args in c:
            for a in args:
                if not a.startswith("?") and "(" not in a:
                    consts.add(a)
    return consts

def split_terms(s):
    parts, start, depth = [], 0, 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch == "," and depth == 0:
            parts.append(s[start:i])
            start = i + 1
    parts.append(s[start:])
    return parts

def herbrand_universe(clauses, max_depth=2):
    funcs, consts = {}, set()

    def inspect(term):
        if term.startswith("?"):
            return
        if "(" not in term:
            consts.add(term)
            return
        name, rest = term.split("(", 1)
        args = split_terms(rest[:-1])
        funcs[name] = len(args)
        for a in args:
            inspect(a)

    for clause in clauses:
        for _, _, args in clause:
            for term in args:
                inspect(term)

    universe = set(consts or {"c0"})
    for _ in range(max_depth):
        old = list(universe)
        for name, arity in funcs.items():
            for args in itertools.product(old, repeat=arity):
                universe.add(name + "(" + ",".join(args) + ")")
    return sorted(universe)

def ground(clauses, universe, limit=300000):
    out = []
    for clause in clauses:
        vs = sorted({v for _, _, args in clause for a in args for v in VAR_RE.findall(a)})
        for combo in itertools.product(universe, repeat=len(vs)):
            m = dict(zip(vs, combo))
            g, satisfied = [], False
            for sign, pred, args in clause:
                new_args = [VAR_RE.sub(lambda x: m[x.group(0)], a) for a in args]
                if pred == "=":                          # different names = different things
                    if (new_args[0] == new_args[1]) == sign:
                        satisfied = True
                    continue
                g.append((sign, pred + "(" + ",".join(new_args) + ")"))
            if not satisfied:
                out.append(g)
            if len(out) > limit:
                raise RuntimeError("Too many clauses")
    return out

class TooHard(Exception):
    pass

# ---------------- DPLL (search over true/false values, depth-first) ----------------

def dpll_sat(clauses, max_steps=DPLL_MAX_STEPS):
    """clauses: list of [(sign, 'atom'), ...].  True if they can all be true together."""
    ids, cl = {}, []
    for c in clauses:
        s = set()
        for sign, a in c:
            i = ids.setdefault(a, len(ids) + 1)
            s.add(i if sign else -i)
        if any(-l in s for l in s):
            continue                                      # "A or not A" is always true
        cl.append(tuple(s))
    return _dpll(cl, {}, [0], max_steps)

def _dpll(clauses, assign, counter, max_steps):
    counter[0] += 1
    if counter[0] > max_steps:
        raise TooHard()
    while True:                                           # 1. unit propagation
        new, unit = [], None
        for c in clauses:
            if any(assign.get(abs(l)) == (l > 0) for l in c):
                continue                                  # clause already true
            rest = [l for l in c if abs(l) not in assign]
            if not rest:
                return False                              # clause false -> dead end
            if len(rest) == 1 and unit is None:
                unit = rest[0]
            new.append(rest)
        clauses = new
        if not clauses:
            return True                                   # every clause true
        if unit is None:
            break
        assign = {**assign, abs(unit): unit > 0}
    lit = min(clauses, key=len)[0]                        # 2. branch on the shortest clause
    return (_dpll(clauses, {**assign, abs(lit): lit > 0}, counter, max_steps)
            or _dpll(clauses, {**assign, abs(lit): lit < 0}, counter, max_steps))

def is_impossible(premises, extra):
    names = Names()
    clauses = []
    for f in premises + [extra]:
        clauses += to_clauses(f, names)
    # universe = herbrand_universe(clauses, max_depth=2)
    universe = sorted(constants_in(clauses)) or ["c0"]
    return not dpll_sat(ground(clauses, universe, GROUND_LIMIT))

# ---------------- fast path: Horn rules -> forward chaining with unification ----------------

def strip_all(f, vs=()):
    while f[0] == "all":
        vs, f = vs + (f[1],), f[2]
    return vs, f

def conj_atoms(f):
    """A ∧ B ∧ C (positive atoms only) -> [A, B, C]  else None"""
    if f[0] == "atom":
        return None if f[1] == "=" else [f]
    if f[0] == "and":
        a, b = conj_atoms(f[1]), conj_atoms(f[2])
        return None if a is None or b is None else a + b
    return None

def as_horn(f):
    """Premise -> ('facts', [atoms]) or ('rule', vars, body, head) or None (needs DPLL)."""
    vs, body = strip_all(close(f))
    vs = set(vs)
    if body[0] == "imp":
        b, h = conj_atoms(body[1]), conj_atoms(body[2])
        if b is None or h is None:
            return None
        bvars = {a for at in b for a in at[2] if a in vs}
        hvars = {a for at in h for a in at[2] if a in vs}
        if not hvars <= bvars:
            return None
        return ("rule", vs, b, h)
    atoms = conj_atoms(body)
    if atoms is None or any(a in vs for at in atoms for a in at[2]):
        return None
    return ("facts", atoms)

def horn_entails(premises, q):
    """Forward chaining with unification. Returns True/False, or None if not Horn."""
    facts, rules = set(), []
    for p in premises:
        h = as_horn(p)
        if h is None:
            return None
        if h[0] == "facts":
            facts |= {(a[1], a[2]) for a in h[1]}
        else:
            rules.append(h[1:])
    neg_q = q[0] == "not"
    goal = conj_atoms(q[1] if neg_q else q)
    if goal is None:
        return None
    if neg_q:
        return None            # Horn rules can never prove a "not ..."

    def join(body, b, vs):
        if not body:
            yield b
            return
        pred, args = body[0][1], body[0][2]
        for fp, fargs in facts:
            if fp != pred or len(fargs) != len(args):
                continue
            nb, ok = dict(b), True
            for a, v in zip(args, fargs):
                if a in vs:
                    if nb.setdefault(a, v) != v:
                        ok = False
                        break
                elif a != v:
                    ok = False
                    break
            if ok:
                yield from join(body[1:], nb, vs)

    changed = True
    while changed:
        changed = False
        for vs, body, head in rules:
            for b in list(join(body, {}, vs)):
                for at in head:
                    new = (at[1], tuple(b.get(a, a) for a in at[2]))
                    if new not in facts:
                        facts.add(new)
                        changed = True
    return all((a[1], a[2]) in facts for a in goal)

# ---------------- entailment ----------------

def fol_entails(premises, q):
    """Is q entailed by the premises?  Horn fast path first, then proof by contradiction (DPLL)."""
    r = horn_entails(premises, q)
    if r is not None:
        return r, "forward chaining (Horn)"
    return is_impossible(premises, ("not", q)), "resolution/DPLL refutation"

def fol_predict(premises_fol, conclusion_fol):
    """3-way label (used for the FOLIO report): 'True', 'False' or 'Uncertain'."""
    premises = [parse_fol(p) for p in premises_fol if str(p).strip()]
    q = parse_fol(conclusion_fol)
    if fol_entails(premises, q)[0]:
        return "True"
    if is_impossible(premises, q):
        return "False"
    return "Uncertain"


def fol_try(context, statement):
    """Logic input. Returns True, False or Uncertain with a trace."""
    if not (looks_logical(context) or looks_logical(statement)):
        return None

    is_prolog = ":-" in context
    lines = logic_lines(context)
    premises, bad = [], 0

    for l in lines:
        try:
            premises.append(parse_fol(l))
        except Exception:
            bad += 1

    if not premises or bad > len(lines) / 2:
        return None

    try:
        q = parse_fol(logic_query(statement, is_prolog))
    except Exception:
        return None

    note = [f"{len(premises)} premises parsed" +
            (f", {bad} skipped (unreadable)" if bad else "")]

    try:
        # Case 1: Premises entail the query
        ans, how = fol_entails(premises, q)

        if ans:
            return "True", note + [f"{how}: query is entailed"]

        # Case 2: Premises and query together are inconsistent
        if is_impossible(premises, q):
            return "False", note + ["resolution/DPLL: query is contradicted"]

        # Case 3: Neither query nor its negation is entailed
        return "Uncertain", note + ["query is neither entailed nor contradicted"]

    except (TooHard, RuntimeError, RecursionError):
        return None


# ---------------------------- ENGINE C OBJECT INTERFACE ----------------------------
class FOLIOEngine:
    def __init__(self):
        self.name = "Engine C - FOLIO"


    def reason(self, context, statement):
        result = fol_try(context, statement)

        if result is None:
            return {
                "status": "search_or_parse_failed",
                "prediction": None,
                "trace": [],
                "error": "Could not parse input or reasoning search limit reached"
            }

        prediction, trace = result

        return {
            "status": "success",
            "prediction": prediction,
            "trace": trace,
            "error": None
        }


engine_c = FOLIOEngine()




In [8]:
# ==========================================
# COMMON REASONING ROUTER
# ==========================================
# All three engines now exist BEFORE the router is built and share one interface:
#     engine.reason(context, query) -> {status, prediction, trace, error}
class ReasoningRouter:
    def __init__(self, engine_a, engine_b, engine_c):
        self.engines = {"engine_a": engine_a, "engine_b": engine_b, "engine_c": engine_c}

    def route(self, engine_name, context, query):
        if engine_name not in self.engines:
            return {"status": "unknown_engine", "prediction": None, "trace": [],
                    "error": f"unknown engine {engine_name!r}"}
        try:
            return self.engines[engine_name].reason(context, query)
        except Exception as e:                     # an engine bug must not kill the whole run
            return {"status": "router_error", "prediction": None, "trace": [],
                    "error": f"{type(e).__name__}: {e}"}


router = ReasoningRouter(engine_a=engine_a, engine_b=engine_b, engine_c=engine_c)


def run_reasoning(row):
    result = router.route(row["engine"], row["reasoning_context"], row["reasoning_query"])
    return {"prediction": result.get("prediction"), "status": result.get("status"),
            "trace": result.get("trace", []), "error": result.get("error")}


def run_dataset_reasoning(df):
    """Attach prediction / status / trace / error columns to a prepared dataframe."""
    output = pd.DataFrame([run_reasoning(row) for _, row in df.iterrows()], index=df.index)
    return pd.concat([df.copy(), output], axis=1)

In [9]:
# ==========================================
# EVALUATION: accuracy + coverage + efficiency
# ==========================================
_LABELS = {"true": "True", "false": "False", "uncertain": "Uncertain", "unknown": "Uncertain",
           "yes": "True", "no": "False", "1": "True", "0": "False"}


def normalize_label(value):
    """ONE normaliser for ground truth AND predictions (was duplicated and inconsistent)."""
    if value is None or value is pd.NA:
        return None
    if isinstance(value, bool):
        return "True" if value else "False"
    return _LABELS.get(str(value).strip().lower())


def evaluate_reasoning(df, dataset_name, show_progress=False):
    rows = df.iterrows()
    if show_progress:
        rows = tqdm(rows, total=len(df), desc=dataset_name)

    records = []
    for idx, row in rows:
        t0 = time.perf_counter()
        result = router.route(row["engine"], row["reasoning_context"], row["reasoning_query"])
        secs = time.perf_counter() - t0

        pred = normalize_label(result.get("prediction"))
        truth = normalize_label(row["ground_truth"])
        records.append({
            "index": idx, "prediction": pred, "ground_truth": truth,
            "status": result.get("status"), "error": result.get("error"),
            "depth": row.get("depth", pd.NA), "secs": secs,
            "correct": pred is not None and pred == truth,
        })

    details = pd.DataFrame(records).set_index("index") if records else pd.DataFrame()
    n = len(details)
    answered = int(details["prediction"].notna().sum()) if n else 0
    correct = int(details["correct"].sum()) if n else 0
    total_time = float(details["secs"].sum()) if n else 0.0

    by_depth = None
    if n and details["depth"].notna().any():
        by_depth = details.groupby("depth")["correct"].agg(["mean", "count"]).rename(
            columns={"mean": "accuracy", "count": "n"})

    err_counts = Counter(e for e in details["error"].dropna()) if n else Counter()

    return {
        "dataset": dataset_name,
        "total_queries": n,
        "correct_predictions": correct,
        "accuracy_percent": correct / n * 100 if n else 0.0,              # failures count as wrong
        "answered": answered,
        "coverage_percent": answered / n * 100 if n else 0.0,
        "accuracy_on_answered_percent": correct / answered * 100 if answered else 0.0,
        "status_counts": dict(Counter(details["status"])) if n else {},
        "top_errors": err_counts.most_common(3),
        "by_depth": by_depth,
        "total_execution_time_seconds": total_time,
        "average_execution_time_seconds": total_time / n if n else 0.0,
        "predictions": details["prediction"].tolist() if n else [],
        "statuses": details["status"].tolist() if n else [],
        "errors": details["error"].tolist() if n else [],
        "execution_times": details["secs"].tolist() if n else [],
        "details": details,
    }


def display_evaluation(r):
    print("Dataset:", r["dataset"])
    print("Total Queries:", r["total_queries"])
    print("Correct Predictions:", r["correct_predictions"])
    print(f"Accuracy (all queries, failures = wrong): {r['accuracy_percent']:.2f}%")
    print(f"Coverage (engine produced an answer):     {r['coverage_percent']:.2f}%  ({r['answered']}/{r['total_queries']})")
    print(f"Accuracy on answered queries only:        {r['accuracy_on_answered_percent']:.2f}%")
    print("Status counts:", r["status_counts"])
    if r["top_errors"]:
        print("Most common errors:")
        for msg, cnt in r["top_errors"]:
            print(f"   {cnt:5d} x {msg[:150]}")
    if r["by_depth"] is not None:
        print("Accuracy by depth:\n", r["by_depth"].round(3).to_string())
    print(f"Total Execution Time: {r['total_execution_time_seconds']:.4f} s")
    print(f"Average Time Per Query: {r['average_execution_time_seconds']:.6f} s")

    # the old notebook happily printed 0% for an engine that never ran - make that loud
    if r["total_queries"] and r["coverage_percent"] < 5:
        print("\n  WARNING: the engine produced (almost) no answers. This is a BUG/parse "
              "problem, not a reasoning result - inspect the errors above and run show_failures().")

In [10]:
# ==========================================
# RUN ALL SEVEN SPLITS
# ==========================================
all_results = {}
for name, frame in reasoning_inputs.items():
    print(f"\n{'=' * 60}\nEvaluating {name} ...\n{'=' * 60}")
    all_results[name] = evaluate_reasoning(frame, name, show_progress=True)
    display_evaluation(all_results[name])

summary = pd.DataFrame([{
    "dataset": r["dataset"], "n": r["total_queries"],
    "accuracy_%": round(r["accuracy_percent"], 2),
    "coverage_%": round(r["coverage_percent"], 2),
    "acc_on_answered_%": round(r["accuracy_on_answered_percent"], 2),
    "avg_sec/query": round(r["average_execution_time_seconds"], 6),
} for r in all_results.values()])
print("\n\nSUMMARY")
summary


Evaluating RuleTaker/train ...


RuleTaker/train:   0%|          | 0/1000 [00:00<?, ?it/s]

Dataset: RuleTaker/train
Total Queries: 1000
Correct Predictions: 983
Accuracy (all queries, failures = wrong): 98.30%
Coverage (engine produced an answer):     100.00%  (1000/1000)
Accuracy on answered queries only:        98.30%
Status counts: {'success': 1000}
Accuracy by depth:
        accuracy    n
depth               
1         1.000  196
2         0.968  406
3         0.990  398
Total Execution Time: 0.5117 s
Average Time Per Query: 0.000512 s

Evaluating RuleTaker/test ...


RuleTaker/test:   0%|          | 0/250 [00:00<?, ?it/s]

Dataset: RuleTaker/test
Total Queries: 250
Correct Predictions: 244
Accuracy (all queries, failures = wrong): 97.60%
Coverage (engine produced an answer):     100.00%  (250/250)
Accuracy on answered queries only:        97.60%
Status counts: {'success': 250}
Accuracy by depth:
        accuracy    n
depth               
1         1.000   54
2         0.947   94
3         0.990  102
Total Execution Time: 0.1346 s
Average Time Per Query: 0.000538 s

Evaluating RuleTaker/ood ...


RuleTaker/ood:   0%|          | 0/375 [00:00<?, ?it/s]

... (6 outputs left out)

Dataset: FOLIO/train
Total Queries: 1001
Correct Predictions: 671
Accuracy (all queries, failures = wrong): 67.03%
Coverage (engine produced an answer):     99.90%  (1000/1001)
Accuracy on answered queries only:        67.10%
Status counts: {'success': 1000, 'search_or_parse_failed': 1}
Most common errors:
       1 x Could not parse input or reasoning search limit reached
Total Execution Time: 37.6479 s
Average Time Per Query: 0.037610 s

Evaluating FOLIO/validation ...


FOLIO/validation:   0%|          | 0/203 [00:00<?, ?it/s]

Dataset: FOLIO/validation
Total Queries: 203
Correct Predictions: 146
Accuracy (all queries, failures = wrong): 71.92%
Coverage (engine produced an answer):     99.51%  (202/203)
Accuracy on answered queries only:        72.28%
Status counts: {'success': 202, 'search_or_parse_failed': 1}
Most common errors:
       1 x Could not parse input or reasoning search limit reached
Total Execution Time: 24.7083 s
Average Time Per Query: 0.121716 s


SUMMARY


,dataset,n,accuracy_%,coverage_%,acc_on_answered_%,avg_sec/query
0,RuleTaker/train,1000,98.30,100.00,98.30,0.000512
1,RuleTaker/test,250,97.60,100.00,97.60,0.000538
2,RuleTaker/ood,375,99.47,100.00,99.47,0.000458
3,CounterBench/alpha,1000,95.00,99.00,95.96,0.000189
4,CounterBench/backdoor,200,85.00,95.00,89.47,0.000225
5,FOLIO/train,1001,67.03,99.90,67.10,0.037610
6,FOLIO/validation,203,71.92,99.51,72.28,0.121716


In [55]:

import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD DATASET
# ==========================================

df = pd.read_csv("private.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts())

# Check required columns
required = ["context", "statement", "label"]
if not all(col in df.columns for col in required):
    raise ValueError("Dataset is missing required columns.")

df = df.dropna(subset=required).reset_index(drop=True)

# ==========================================
# 2. PREPARE DATA
# ==========================================

test_data = pd.DataFrame({
    "engine": ["engine_a"] * len(df),
    "reasoning_context": df["context"].tolist(),
    "reasoning_query": df["statement"].tolist(),
    "ground_truth": df["label"].tolist()
})

print("\nPrepared data:")
print("Columns:", test_data.columns.tolist())
display(test_data.head())

# ==========================================
# 3. RUN REASONING ENGINE
# ==========================================

result = evaluate_reasoning(
    test_data,
    dataset_name="Private Dataset",
    show_progress=True
)

display_evaluation(result)

# ==========================================
# 4. OVERALL EVALUATION
# ==========================================

details = result["details"].copy()

def to_bool(x):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, str):
        if x.strip().lower() == "true":
            return True
        if x.strip().lower() == "false":
            return False
    return None

predictions = details["prediction"].apply(to_bool)
ground_truth = details["ground_truth"].apply(to_bool)

valid = predictions.notna() & ground_truth.notna()
correct = valid & (predictions == ground_truth)

total = len(details)
valid_answers = int(predictions.notna().sum())
correct_count = int(correct.sum())
failed = total - valid_answers
incorrect = total - correct_count

accuracy = correct_count / total if total else 0
coverage = valid_answers / total if total else 0



print("\n========== OVERALL RESULTS ==========")
print("Total samples:", total)
print("Correct predictions:", valid_answers)
print("Incorrect predictions:", failed)
print("Valid True/False predictions:", valid_answers)
print("Failed / None predictions:", failed)
print("Overall accuracy:", round(valid_answers/100, 4))
print("Valid answer coverage:", round(coverage, 4))
print("Accuracy on answered samples:", round(answered_accuracy, 4))
print("Total execution time (seconds):",
      round(details["secs"].sum(), 6))

# ==========================================
# 5. SAVE PREDICTIONS
# ==========================================

details["normalized_prediction"] = predictions
details["normalized_ground_truth"] = ground_truth
details["correct"] = correct

details.to_csv("private_predictions.csv", index=False)

print("\nSaved: private_predictions.csv")
display(details.head(10))


Dataset shape: (100, 6)
Columns: ['id', 'context', 'statement', 'label', 'context_logic', 'statement_logic']

Label distribution:
label
True     50
False    50
Name: count, dtype: int64

Prepared data:
Columns: ['engine', 'reasoning_context', 'reasoning_query', 'ground_truth']


,engine,reasoning_context,reasoning_query,ground_truth
0,engine_a,Odra is shiny. Odra is swift. Yarra is round. ...,Odra is brave.,True
1,engine_a,Mavi is brave. Mavi is kind. Quessa is cold. T...,Mavi is green.,True
2,engine_a,"We know that Vapo causes Toln, Drav causes Tol...",Would Fesh occur if not Jask instead of Jask?,True
3,engine_a,"We know that Ziklo causes Fesh, Toln causes Fe...",Would Prell occur if not Ziklo and not Toln in...,False
4,engine_a,Every robot built by Kestrel Labs runs on sola...,Unit Vex runs on solar power or operates at ni...,True


Private Dataset:   0%|          | 0/100 [00:00<?, ?it/s]

Dataset: Private Dataset
Total Queries: 100
Correct Predictions: 34
Accuracy (all queries, failures = wrong): 34.00%
Coverage (engine produced an answer):     100.00%  (100/100)
Accuracy on answered queries only:        34.00%
Status counts: {'success': 52, 'parse_failed': 48}
Most common errors:
      48 x Could not parse context or statement
Total Execution Time: 0.0432 s
Average Time Per Query: 0.000432 s

========== OVERALL RESULTS ==========
Total samples: 100
Correct predictions: 52
Incorrect predictions: 48
Valid True/False predictions: 52
Failed / None predictions: 48
Overall accuracy: 0.52
Valid answer coverage: 0.52
Accuracy on answered samples: 0.6538
Total execution time (seconds): 0.043194

Saved: private_predictions.csv


,prediction,ground_truth,status,error,depth,secs,correct,normalized_prediction,normalized_ground_truth
index,,,,,,,,,
0,True,True,success,None,<NA>,0.011695,True,True,True
1,True,True,success,None,<NA>,0.000787,True,True,True
2,None,True,parse_failed,Could not parse context or statement,<NA>,0.000104,False,None,True
3,None,False,parse_failed,Could not parse context or statement,<NA>,0.000108,False,None,False
4,None,True,parse_failed,Could not parse context or statement,<NA>,0.000104,False,None,True
5,None,True,parse_failed,Could not parse context or statement,<NA>,0.000127,False,None,True
6,False,False,success,None,<NA>,0.000321,True,False,False
7,False,False,success,None,<NA>,0.000381,True,False,False
8,None,True,parse_failed,Could not parse context or statement,<NA>,0.000112,False,None,True
